# NB08 · Faster R-CNN frente a YOLO11s y YOLO26s — pollos

**Proyecto:** AgroUTMACH PIX  
**Caso del libro:** detección y conteo de pollos  
**Arquitectura adicional:** Faster R-CNN con ResNet-50-FPN V2  
**Entorno obligatorio:** GPU NVIDIA  
**Recomendado:** GPU L4 de Google Colab  
**Tiempo estimado:** L4, 2–3.5 h; T4, 4–6 h  
**CPU:** no recomendada (aprox. 20–40 h)

El conjunto contiene escenas densas con cientos de pollos por imagen. Por ello se utiliza batch 1, hasta 1.000 detecciones por imagen, evaluación COCO común para las tres arquitecturas y guardado por época.

Este notebook:

- requiere que NB07 haya terminado;
- reutiliza YOLO11s y YOLO26s;
- entrena Faster R-CNN durante un máximo de 12 épocas;
- reanuda automáticamente desde la última época guardada;
- compara mAP, recall, error de conteo, tamaño y velocidad;
- genera figuras a 300 dpi para el capítulo 6.

> Debido a que el proyecto dispone de train/val, la comparación final sobre `val` debe presentarse como evaluación comparativa interna, no como prueba externa independiente.


In [ ]:
%pip install -q torchmetrics pycocotools seaborn ultralytics==8.4.172 pyyaml


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import json
import math
import os

# Debe definirse antes de inicializar CUDA; reduce fragmentación del asignador.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_V2_Weights,
    fasterrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. En Colab seleccione Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución > GPU L4 o T4.'
    )

DEVICE = torch.device('cuda:0')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)


## 1. Rutas y requisitos


In [ ]:
ROOT = Path('/content/drive/MyDrive/AgroUTMACH_PIX')
DATASET_DRIVE = ROOT / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MODELS_DIR = ROOT / '03_MODELOS'
RESULTS_DIR = ROOT / '05_RESULTADOS'
NB08_RESULTS = RESULTS_DIR / 'NB08_FASTER_RCNN_BENCHMARK'

YOLO11_MODEL = MODELS_DIR / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
YOLO26_MODEL = MODELS_DIR / 'AgroUTMACH_PIX_YOLO26s_PIO_best.pt'
FRCNN_MODEL = MODELS_DIR / 'AgroUTMACH_PIX_FasterRCNN_ResNet50FPN_best.pt'
CHECKPOINT = NB08_RESULTS / 'checkpoint_last.pt'
HISTORY_CSV = NB08_RESULTS / 'historial_entrenamiento.csv'

LOCAL_DATA = Path('/content/PIO_NB08_LOCAL')
LOCAL_YAML = LOCAL_DATA / 'dataset_colab.yaml'

NB08_RESULTS.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    DATASET_DRIVE / 'images' / 'train', DATASET_DRIVE / 'images' / 'val',
    DATASET_DRIVE / 'labels' / 'train', DATASET_DRIVE / 'labels' / 'val',
    YOLO11_MODEL, YOLO26_MODEL,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Faltan recursos. Ejecute primero NB07 y verifique el dataset:\n' + '\n'.join(missing)
    )

print('YOLO11s:', YOLO11_MODEL)
print('YOLO26s:', YOLO26_MODEL)
print('Resultados NB08:', NB08_RESULTS)


## 2. Copia local y lectura de anotaciones YOLO


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)
shutil.copytree(DATASET_DRIVE / 'images', LOCAL_DATA / 'images')
shutil.copytree(DATASET_DRIVE / 'labels', LOCAL_DATA / 'labels')

extra_classes = LOCAL_DATA / 'labels' / 'train' / 'classes.txt'
if extra_classes.exists():
    extra_classes.unlink()

dataset_cfg = {
    'path': str(LOCAL_DATA), 'train': 'images/train', 'val': 'images/val',
    'nc': 1, 'names': ['Pollo'],
}
LOCAL_YAML.write_text(
    yaml.safe_dump(dataset_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8'
)


class ChickenDetectionDataset(Dataset):
    def __init__(self, root, split, augment=False):
        self.root = Path(root)
        self.split = split
        self.augment = augment
        self.images = sorted(
            path for path in (self.root / 'images' / split).iterdir()
            if path.is_file() and path.suffix.lower() in IMG_EXTS
        )
        if not self.images:
            raise RuntimeError(f'No se encontraron imágenes en {split}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        label_path = self.root / 'labels' / self.split / f'{image_path.stem}.txt'
        image = Image.open(image_path).convert('RGB')
        width, height = image.size
        boxes = []
        if label_path.exists():
            for line in label_path.read_text(encoding='utf-8').splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                _, xc, yc, bw, bh = map(float, parts[:5])
                x1 = max(0.0, (xc - bw / 2) * width)
                y1 = max(0.0, (yc - bh / 2) * height)
                x2 = min(float(width), (xc + bw / 2) * width)
                y2 = min(float(height), (yc + bh / 2) * height)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.ones((len(boxes),), dtype=torch.int64)
        image = TF.to_tensor(image)

        if self.augment and random.random() < 0.5:
            image = TF.hflip(image)
            if len(boxes):
                old_x1 = boxes[:, 0].clone()
                old_x2 = boxes[:, 2].clone()
                boxes[:, 0] = width - old_x2
                boxes[:, 2] = width - old_x1

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([index]),
            'area': ((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = ChickenDetectionDataset(LOCAL_DATA, 'train', augment=True)
val_dataset = ChickenDetectionDataset(LOCAL_DATA, 'val', augment=False)
assert len(train_dataset) == 1035 and len(val_dataset) == 452

train_loader = DataLoader(
    train_dataset, batch_size=1, shuffle=True, num_workers=2,
    pin_memory=True, collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False, num_workers=2,
    pin_memory=True, collate_fn=collate_fn,
)
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')


## 3. Modelo Faster R-CNN adaptado a escenas densas


In [ ]:
def build_faster_rcnn():
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=640,
        max_size=960,
        box_detections_per_img=1000,
        rpn_pre_nms_top_n_train=3000,
        rpn_post_nms_top_n_train=2000,
        rpn_pre_nms_top_n_test=3000,
        rpn_post_nms_top_n_test=1500,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    return model


model = build_faster_rcnn().to(DEVICE)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {trainable_parameters / 1e6:.2f} millones')


## 4. Entrenamiento reanudable


In [ ]:
MAX_EPOCHS = 12
PATIENCE = 3
LEARNING_RATE = 0.0025
WEIGHT_DECAY = 0.0005
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, momentum=0.9, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.3)
scaler = torch.amp.GradScaler('cuda', enabled=True)

start_epoch = 0
best_map50 = -1.0
epochs_without_improvement = 0
history = []

if CHECKPOINT.exists():
    state = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    scheduler.load_state_dict(state['scheduler_state'])
    scaler.load_state_dict(state['scaler_state'])
    start_epoch = int(state['epoch']) + 1
    best_map50 = float(state['best_map50'])
    epochs_without_improvement = int(state['epochs_without_improvement'])
    history = state.get('history', [])
    del state
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Reanudando desde la época {start_epoch + 1}')
    if epochs_without_improvement >= PATIENCE:
        print('La parada temprana ya se alcanzó; no se repetirá el entrenamiento.')
        start_epoch = MAX_EPOCHS


def evaluate_frcnn(model, loader, score_for_count=0.25):
    model.eval()
    metric = MeanAveragePrecision(
        box_format='xyxy', iou_type='bbox',
        max_detection_thresholds=[100, 300, 1000],
    )
    count_errors = []
    count_biases = []
    start = time.perf_counter()
    with torch.inference_mode():
        for images, targets in loader:
            images_gpu = [image.to(DEVICE, non_blocking=True) for image in images]
            predictions = model(images_gpu)
            preds_cpu = []
            targets_cpu = []
            for pred, target in zip(predictions, targets):
                pred_cpu = {key: value.detach().cpu() for key, value in pred.items()}
                target_cpu = {
                    'boxes': target['boxes'].cpu(),
                    'labels': target['labels'].cpu(),
                }
                preds_cpu.append(pred_cpu)
                targets_cpu.append(target_cpu)
                predicted_count = int((pred_cpu['scores'] >= score_for_count).sum())
                true_count = int(len(target_cpu['boxes']))
                count_errors.append(abs(predicted_count - true_count))
                count_biases.append(predicted_count - true_count)
            metric.update(preds_cpu, targets_cpu)
    elapsed = time.perf_counter() - start
    result = metric.compute()
    return {
        'mAP50_95': float(result['map']),
        'mAP50': float(result['map_50']),
        'mAP75': float(result['map_75']),
        'mAR1000': float(result['mar_1000']),
        'count_mae': float(np.mean(count_errors)),
        'count_bias': float(np.mean(count_biases)),
        'inference_ms_image': float(elapsed * 1000 / len(loader.dataset)),
    }


training_start = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    epoch_losses = []
    epoch_start = time.time()
    for step, (images, targets) in enumerate(train_loader, start=1):
        images = [image.to(DEVICE, non_blocking=True) for image in images]
        targets = [
            {key: value.to(DEVICE, non_blocking=True) for key, value in target.items()}
            for target in targets
        ]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=True):
            loss_dict = model(images, targets)
            loss = sum(loss_value for loss_value in loss_dict.values())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(float(loss.detach().cpu()))
        if step % 100 == 0:
            print(
                f'Época {epoch + 1}/{MAX_EPOCHS} · paso {step}/{len(train_loader)} '
                f'· pérdida {np.mean(epoch_losses[-100:]):.4f}'
            )

    scheduler.step()
    val_metrics = evaluate_frcnn(model, val_loader)
    row = {
        'epoch': epoch + 1,
        'train_loss': float(np.mean(epoch_losses)),
        'epoch_minutes': float((time.time() - epoch_start) / 60),
        'learning_rate': float(optimizer.param_groups[0]['lr']),
        **val_metrics,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
    print(row)

    improved = val_metrics['mAP50'] > best_map50 + 1e-4
    if improved:
        best_map50 = val_metrics['mAP50']
        epochs_without_improvement = 0
        torch.save(model.state_dict(), FRCNN_MODEL)
        print('Nuevo mejor modelo guardado.')
    else:
        epochs_without_improvement += 1

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_map50': best_map50,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
    }, CHECKPOINT)

    if epochs_without_improvement >= PATIENCE:
        print(f'Parada temprana después de {PATIENCE} épocas sin mejora.')
        break

training_minutes_session = (time.time() - training_start) / 60
print(f'Sesión de entrenamiento: {training_minutes_session:.1f} min')
print('Mejor mAP@50 de validación:', best_map50)


## 5. Evaluación común de Faster R-CNN y YOLO


In [ ]:
if not FRCNN_MODEL.exists():
    raise FileNotFoundError('No se encontró el mejor modelo Faster R-CNN.')


def liberar_gpu(*nombres):
    """Libera referencias globales y fuerza la devolución de memoria CUDA."""
    for nombre in nombres:
        objeto = globals().pop(nombre, None)
        if isinstance(objeto, torch.nn.Module):
            objeto.to('cpu')
        del objeto
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()


def memoria_libre_gpu():
    return torch.cuda.mem_get_info()[0] / 1024**3


liberar_gpu(
    'model', 'optimizer', 'scheduler', 'scaler', 'loss_dict', 'loss',
    'images', 'targets', 'images_gpu', 'predictions', 'state',
    'frcnn_best', 'yolo', 'outputs',
)
print(f'Memoria GPU libre al iniciar la evaluación: {memoria_libre_gpu():.2f} GB')


def evaluate_yolo_common(label, model_path, score_for_count=0.25):
    """Evalúa YOLO desde una carpeta, en flujo y con lote real de una imagen."""
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Iniciando {label} · GPU libre: {memoria_libre_gpu():.2f} GB')

    yolo = YOLO(str(model_path))
    metric = MeanAveragePrecision(
        box_format='xyxy', iou_type='bbox',
        max_detection_thresholds=[100, 300, 1000],
    )
    count_errors = []
    count_biases = []
    image_paths = val_dataset.images
    index_by_name = {path.name: index for index, path in enumerate(image_paths)}
    validation_folder = LOCAL_DATA / 'images' / 'val'
    start = time.perf_counter()

    # IMPORTANTE: una carpeta es una fuente de archivos en flujo. Una lista de
    # 452 rutas puede ser interpretada por Ultralytics como un lote en memoria.
    outputs = yolo.predict(
        source=str(validation_folder),
        imgsz=960,
        conf=0.001,
        iou=0.7,
        max_det=1000,
        batch=1,
        stream=True,
        device=0,
        verbose=False,
    )

    processed_names = set()
    for output in outputs:
        image_name = Path(output.path).name
        if image_name not in index_by_name:
            raise RuntimeError(f'{label}: imagen inesperada en la salida: {image_name}')
        index = index_by_name[image_name]
        target = val_dataset[index][1]
        processed_names.add(image_name)

        if output.boxes is None:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            scores = torch.empty((0,), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = output.boxes.xyxy.detach().cpu().float()
            scores = output.boxes.conf.detach().cpu().float()
            labels = torch.ones((len(boxes),), dtype=torch.int64)

        pred = {'boxes': boxes, 'scores': scores, 'labels': labels}
        truth = {'boxes': target['boxes'], 'labels': target['labels']}
        metric.update([pred], [truth])
        predicted_count = int((scores >= score_for_count).sum())
        true_count = int(len(target['boxes']))
        count_errors.append(abs(predicted_count - true_count))
        count_biases.append(predicted_count - true_count)

    if len(processed_names) != len(image_paths):
        missing_names = sorted(set(index_by_name) - processed_names)
        raise RuntimeError(
            f'{label}: se procesaron {len(processed_names)} de {len(image_paths)} imágenes. '
            f'Faltantes iniciales: {missing_names[:5]}'
        )

    elapsed = time.perf_counter() - start
    result = metric.compute()
    parameters = sum(p.numel() for p in yolo.model.parameters())
    row = {
        'model': label,
        'mAP50_95': float(result['map']),
        'mAP50': float(result['map_50']),
        'mAP75': float(result['map_75']),
        'mAR1000': float(result['mar_1000']),
        'count_mae': float(np.mean(count_errors)),
        'count_bias': float(np.mean(count_biases)),
        'inference_ms_image': float(elapsed * 1000 / len(image_paths)),
        'parameters_millions': float(parameters / 1e6),
        'model_size_mb': float(model_path.stat().st_size / 1024**2),
    }

    # El predictor conserva una referencia propia al modelo; se retiran ambas.
    if getattr(yolo, 'predictor', None) is not None:
        predictor_model = getattr(yolo.predictor, 'model', None)
        if hasattr(predictor_model, 'to'):
            predictor_model.to('cpu')
        yolo.predictor = None
    yolo.model.to('cpu')
    del outputs, yolo, metric, result
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f'{label} evaluado correctamente · GPU libre: {memoria_libre_gpu():.2f} GB')
    return row


# Los YOLO se evalúan primero sobre una GPU completamente limpia.
comparison_rows = []
for label, model_path in [
    ('YOLO11s', YOLO11_MODEL),
    ('YOLO26s', YOLO26_MODEL),
]:
    comparison_rows.append(evaluate_yolo_common(label, model_path))
    pd.DataFrame(comparison_rows).to_csv(
        NB08_RESULTS / 'comparacion_FasterRCNN_YOLO_parcial.csv', index=False
    )

# Faster R-CNN se carga al final; después ya no se necesita cargar otro modelo.
gc.collect()
torch.cuda.empty_cache()
print(f'Iniciando Faster R-CNN · GPU libre: {memoria_libre_gpu():.2f} GB')
frcnn_best = build_faster_rcnn().to(DEVICE)
frcnn_best.load_state_dict(
    torch.load(FRCNN_MODEL, map_location=DEVICE, weights_only=True)
)
frcnn_metrics = evaluate_frcnn(frcnn_best, val_loader)
frcnn_metrics.update({
    'model': 'Faster R-CNN',
    'parameters_millions': sum(p.numel() for p in frcnn_best.parameters()) / 1e6,
    'model_size_mb': FRCNN_MODEL.stat().st_size / 1024**2,
})
frcnn_best.to('cpu')
del frcnn_best
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

comparison = pd.DataFrame(comparison_rows + [frcnn_metrics])
comparison.to_csv(NB08_RESULTS / 'comparacion_FasterRCNN_YOLO.csv', index=False)
display(comparison.round(4))


## 6. Figuras profesionales para el capítulo 6


In [ ]:
sns.set_theme(style='whitegrid', context='talk')
palette = {'YOLO11s': '#1F5C3D', 'YOLO26s': '#D18B2C', 'Faster R-CNN': '#315A91'}

long_metrics = comparison.melt(
    id_vars='model', value_vars=['mAP50', 'mAP50_95', 'mAR1000'],
    var_name='metric', value_name='value',
)
labels = {'mAP50': 'mAP@50', 'mAP50_95': 'mAP@50–95', 'mAR1000': 'mAR@1000'}
long_metrics['metric'] = long_metrics['metric'].map(labels)

fig, ax = plt.subplots(figsize=(10.8, 5.8))
sns.barplot(data=long_metrics, x='metric', y='value', hue='model', palette=palette, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=8.5)
ax.set_ylim(0, 1.06)
ax.set_xlabel('')
ax.set_ylabel('Valor')
ax.set_title('Detección densa de pollos: comparación común')
ax.legend(title='Modelo', frameon=True, loc='lower left')
fig.savefig(NB08_RESULTS / 'Figura_NB08_metricas_deteccion.png', dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
sns.barplot(data=comparison, x='model', y='count_mae', hue='model', palette=palette,
            legend=False, ax=axes[0])
axes[0].set_title('Error absoluto de conteo')
axes[0].set_xlabel('')
axes[0].set_ylabel('MAE (pollos/imagen)')
axes[0].tick_params(axis='x', rotation=12)

for _, row in comparison.iterrows():
    axes[1].scatter(row['inference_ms_image'], row['mAP50_95'], s=170,
                    color=palette[row['model']], edgecolor='black', linewidth=0.7)
    axes[1].annotate(row['model'], (row['inference_ms_image'], row['mAP50_95']),
                     xytext=(6, 5), textcoords='offset points', fontsize=9)
axes[1].set_title('Precisión frente a velocidad')
axes[1].set_xlabel('Tiempo observado (ms/imagen)')
axes[1].set_ylabel('mAP@50–95')

fig.savefig(NB08_RESULTS / 'Figura_NB08_conteo_velocidad.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Trazabilidad y cierre


In [ ]:
winner_detection = comparison.sort_values('mAP50_95', ascending=False).iloc[0]
winner_count = comparison.sort_values('count_mae', ascending=True).iloc[0]
conclusion = {
    'best_detection_model': winner_detection['model'],
    'best_detection_mAP50_95': float(winner_detection['mAP50_95']),
    'best_counting_model': winner_count['model'],
    'best_counting_MAE': float(winner_count['count_mae']),
    'evaluation_scope': 'Comparación interna sobre el conjunto de validación común.',
    'count_threshold': 0.25,
    'warning': 'La precisión, el conteo y la velocidad deben interpretarse conjuntamente.',
}
(NB08_RESULTS / 'conclusion_benchmark.json').write_text(
    json.dumps(conclusion, ensure_ascii=False, indent=2), encoding='utf-8'
)

environment = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'gpu': GPU_NAME,
    'gpu_memory_gb': round(GPU_MEMORY_GB, 2),
    'models': ['YOLO11s', 'YOLO26s', 'Faster R-CNN ResNet-50-FPN V2'],
    'seed': SEED,
}
(NB08_RESULTS / 'entorno_ejecucion.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('=' * 72)
print('NB08 FINALIZADO CORRECTAMENTE')
print('Mejor detección:', winner_detection['model'])
print('Mejor conteo:', winner_count['model'])
print('Resultados:', NB08_RESULTS)
print('=' * 72)


## Resultado esperado

Al terminar debe aparecer **`NB08 FINALIZADO CORRECTAMENTE`**.  
Si Colab se desconecta, vuelva a ejecutar desde el inicio: el entrenamiento se reanudará desde `checkpoint_last.pt`.

**Corrección de memoria:** la evaluación libera Faster R-CNN antes de cargar cada YOLO y procesa las imágenes secuencialmente con `batch=1` y `stream=True`.

**Evaluación de memoria v2:** YOLO11s y YOLO26s se procesan primero desde la carpeta de validación con un lote real de una imagen. Faster R-CNN se carga únicamente después de liberar ambos modelos YOLO.
